# AI-Assisted, Vibe, and Agentic Coding
## An example: developing an RNA alignment algorithm using RNA-FM

This notebook compares three styles of AI-enabled programming using a single computational biology problem:

> **Develop a pairwise RNA sequence alignment algorithm that uses nucleotide-level embeddings from RNA-FM to improve alignment over sequence-only methods.**

The goal is not to implement a production-ready RNA-FM alignment system here. Instead, the notebook uses this problem to make the distinction between **AI-assisted coding**, **vibe coding**, and **agentic coding** concrete.

---

### Running example

For two RNA sequences

$X=x_1,\ldots,x_n,\qquad Y=y_1,\ldots,y_m$

suppose RNA-FM produces contextual nucleotide embeddings

$E_X=(e_1,\ldots,e_n), \qquad E_Y=(f_1,\ldots,f_m)$

A natural research question is whether an alignment score that incorporates embedding similarity can improve alignment of evolutionarily distant RNAs.

## 1. At a glance

|  | **AI-assisted coding** | **Vibe coding** | **Agentic coding** |
|---|---|---|---|
| Human's main role | Algorithm designer + programmer | Problem describer + evaluator | Researcher + supervisor |
| AI's main role | Coding assistant | Primary implementer | Autonomous research/coding agent |
| Who designs the alignment algorithm? | Mostly human | Often AI | Human sets goals; AI may propose and test alternatives |
| Who writes most code? | Human + AI | Mostly AI | Mostly AI |
| Must the human understand implementation details? | Usually yes | Not necessarily | Should understand the design and evidence |
| AI runs experiments? | Usually when asked | Often | Autonomously |
| AI diagnoses failures? | Helps the human | Prompt-driven | Yes |
| AI changes the algorithm based on results? | Human decides | User asks for changes | Can propose, implement, and evaluate changes |
| Best use | Careful implementation | Rapid exploration/prototyping | Larger research-development workflows |

A useful shorthand is:

- **AI-assisted coding:** *I know the algorithm; help me code it.*
- **Vibe coding:** *I know what I want the program to do; build something that does it.*
- **Agentic coding:** *I know the research objective and constraints; systematically develop, test, and improve an algorithm to address it.*

## 2. A possible RNA-FM-based alignment score

A conventional alignment algorithm uses nucleotide substitution scores and gap penalties.

With RNA-FM, one could augment the nucleotide score with embedding similarity. For example,

$$
S(i,j)
=
\alpha S_{\mathrm{nt}}(x_i,y_j)
+
(1-\alpha)\cos(e_i,f_j).
$$

Here:

- $S_{\mathrm{nt}}$ is a conventional nucleotide match/mismatch score;
- $\cos(e_i,f_j)$ is cosine similarity between RNA-FM embeddings;
- $\alpha\in[0,1]$ controls the contribution of sequence identity versus language-model similarity.

The exact scoring function is a **research choice**, not something guaranteed to work simply because RNA-FM embeddings are available.

In [ ]:
import numpy as np

def cosine_similarity_matrix(E1, E2):
    """Compute pairwise cosine similarities between two embedding matrices."""
    E1 = np.asarray(E1, dtype=float)
    E2 = np.asarray(E2, dtype=float)

    E1 = E1 / np.linalg.norm(E1, axis=1, keepdims=True)
    E2 = E2 / np.linalg.norm(E2, axis=1, keepdims=True)

    return E1 @ E2.T

The function above is intentionally simple. In a real project, embeddings would be produced by RNA-FM and the scoring matrix would feed into a dynamic-programming alignment algorithm.

# 3. AI-assisted coding

In **AI-assisted coding**, the human remains the primary algorithm designer.

The researcher might decide in advance:

1. use RNA-FM nucleotide embeddings;
2. compute pairwise cosine similarity;
3. combine embedding similarity with nucleotide identity;
4. perform global alignment using Needleman-Wunsch;
5. use affine gap penalties;
6. benchmark against sequence-only alignment.

The human might then ask an AI system:

> Write a PyTorch function that takes two RNA sequences, obtains their RNA-FM embeddings, computes the cosine-similarity matrix, and uses it in Needleman-Wunsch alignment.

The AI is helping to implement a design that the researcher already understands.

### Example recurrence

For a simple global alignment with linear gap penalty \(g\),

$$
D(i,j)=\max
\begin{cases}
D(i-1,j-1)+S(i,j)\\
D(i-1,j)+g\\
D(i,j-1)+g
\end{cases}
$$

The researcher decides that this is the desired algorithm. AI may help translate the recurrence into reliable code.

In [ ]:
def needleman_wunsch(score_matrix, gap=-1.0):
    """Simple global alignment using a precomputed pairwise score matrix."""
    n, m = score_matrix.shape
    D = np.zeros((n + 1, m + 1))
    traceback = np.empty((n + 1, m + 1), dtype=object)

    for i in range(1, n + 1):
        D[i, 0] = i * gap
        traceback[i, 0] = "up"

    for j in range(1, m + 1):
        D[0, j] = j * gap
        traceback[0, j] = "left"

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            choices = {
                "diag": D[i-1, j-1] + score_matrix[i-1, j-1],
                "up":   D[i-1, j] + gap,
                "left": D[i, j-1] + gap,
            }
            move = max(choices, key=choices.get)
            D[i, j] = choices[move]
            traceback[i, j] = move

    return D, traceback

### Typical workflow

```text
Human formulates algorithm
          ↓
AI helps implement
          ↓
Human inspects code
          ↓
Human runs tests
          ↓
Human modifies algorithm
```

The key characteristic is that **the human still owns the implementation logic and algorithmic decisions**.

# 4. Vibe coding

In **vibe coding**, the interaction starts from desired behavior rather than a carefully specified algorithm.

A researcher might say:

> Build me an RNA sequence alignment program using RNA-FM. It should align homologous RNAs better than conventional sequence alignment, particularly when sequence identity is low.

The AI might then decide on its own to:

- obtain RNA-FM embeddings;
- calculate cosine similarities;
- implement dynamic programming;
- choose gap penalties;
- create a command-line interface;
- visualize alignments;
- test several examples.

The researcher mainly evaluates whether the resulting behavior looks useful.

### Example interaction

```text
Researcher:
"Build an RNA alignment tool using RNA-FM."

AI:
[generates implementation]

Researcher:
"The low-identity alignments don't look very good. Improve them."

AI:
[changes scoring function]

Researcher:
"Try combining nucleotide identity with RNA-FM similarity."

AI:
[changes implementation]

Researcher:
"This seems better. Add local alignment and make comparison plots."
```

The workflow becomes:

```text
idea
 ↓
prompt
 ↓
AI generates implementation
 ↓
run
 ↓
observe output
 ↓
describe what seems wrong
 ↓
AI changes implementation
 ↓
repeat
```

The researcher may end up with a useful program without closely inspecting the dynamic-programming recurrence, normalization, gap logic, batching, or edge cases.

### Main benefit

Vibe coding can be excellent for **rapid exploration**.

### Main scientific risk

A result can look convincing while still containing:

- algorithmic bugs;
- inappropriate benchmarks;
- data leakage;
- poorly chosen baselines;
- invalid evaluation;
- biologically misleading assumptions.

For scientific computing, visual plausibility is not enough.

# 5. Agentic coding

In **agentic coding**, the AI takes responsibility for a larger multi-step development loop.

A researcher could specify:

> Develop and evaluate an RNA pairwise alignment method based on RNA-FM embeddings. Compare several ways of incorporating RNA-FM information with conventional sequence similarity. Benchmark against appropriate sequence-only baselines. Optimize hyperparameters using training data and evaluate on held-out RNA families. Add tests and summarize the results.

The agent can then perform a sequence of actions with relatively little intervention.

### Agentic workflow

```text
                   RESEARCHER
                       │
              goal + constraints
                       ↓
                 AI CODING AGENT
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
      inspect       design       inspect
      RNA-FM       methods       datasets
          │            │            │
          └────────────┼────────────┘
                       ↓
                 implement v1
                       ↓
                   run tests
                       ↓
                 run benchmark
                       ↓
                analyze results
                       ↓
              ┌────────┴────────┐
              ↓                 ↓
           problem             good
              ↓                 ↓
       modify method         report
              │
              └──────→ repeat
```

The defining feature is the **plan → act → observe → revise** loop.

## 6. What might the agent investigate?

An agent might automatically compare several scoring functions.

### Embedding-only score

$$
S_1(i,j)=\cos(e_i,f_j)
$$

### Sequence + embedding score

$$
S_2(i,j)
=
\alpha I(x_i=x_j)
+
(1-\alpha)\cos(e_i,f_j)
$$

### Learned score

$$
S_3(i,j)
=
\operatorname{MLP}
\left(
[e_i,f_j,e_i\odot f_j]
\right).
$$

The agent could then evaluate:

- linear versus affine gap penalties;
- global versus local alignment;
- different values of \(\alpha\);
- different embedding layers;
- dimensionality reduction of RNA-FM representations;
- runtime and memory usage;
- accuracy across RNA families and sequence-identity ranges.

In [ ]:
def combined_score(nt1, nt2, emb_similarity, alpha=0.5,
                   match_score=1.0, mismatch_score=-1.0):
    """Illustrative hybrid nucleotide + embedding score."""
    seq_score = match_score if nt1 == nt2 else mismatch_score
    return alpha * seq_score + (1 - alpha) * emb_similarity

In an agentic workflow, the AI could write this function, integrate it into the aligner, sweep over `alpha`, run benchmarks, diagnose failures, and revise the method.

The researcher is supervising a **research-development process**, not merely asking for individual code fragments.

# 7. The key distinction: what is being delegated?

The three approaches are easiest to distinguish by asking what intellectual and operational work is delegated.

### AI-assisted coding

```text
Human:
"Use RNA-FM cosine similarity +
 Needleman-Wunsch with affine gaps."

AI:
"Here is the implementation."
```

The human specifies the algorithm.

### Vibe coding

```text
Human:
"Make an RNA alignment tool using RNA-FM."

AI:
"Here is one."

Human:
"It doesn't work well on these examples.
Improve it."
```

The human specifies desired behavior and evaluates outcomes.

### Agentic coding

```text
Human:
"Investigate whether RNA-FM can improve
alignment of distantly related RNAs.
Develop and benchmark an approach."

AI agent:
design → implement → test → benchmark
   ↑                         ↓
   └──── revise ← analyze ───┘
```

The human specifies the scientific objective and constraints, while the AI executes much of the development loop.

# 8. Vibe coding and agentic coding are not the same dimension

It is tempting to describe the progression as

```text
traditional → AI-assisted → vibe → agentic
```

but that is misleading.

**Vibe coding** mainly concerns how much the programmer works directly with and understands the implementation.

**Agentic coding** mainly concerns how much autonomy the AI has to plan, execute tools, test, and iterate.

Therefore, one can have:

### Agentic vibe coding

The user says:

> Build an RNA-FM alignment program and make it work well.

The agent autonomously develops it, and the user barely examines the implementation.

### Agentic scientific programming

The researcher gives the agent a precise scientific objective, allows it to implement and benchmark alternatives, but carefully reviews:

- algorithmic assumptions;
- training/test separation;
- benchmark design;
- statistical evidence;
- source code changes;
- biological interpretation.

The AI can have substantial autonomy **without the researcher giving up scientific understanding**.

# 9. What students still need to understand

Agentic coding does not eliminate the need to learn computational biology.

To judge whether an AI-developed RNA alignment method is scientifically sound, students still need to understand:

- dynamic programming;
- global and local sequence alignment;
- affine gap penalties;
- substitution scoring;
- RNA sequence and structure;
- representation learning and embeddings;
- train/validation/test separation;
- homology and RNA-family structure;
- benchmark construction;
- data leakage;
- runtime complexity;
- statistical evaluation.

In fact, greater AI autonomy can make these concepts **more important**, because the student must evaluate decisions they did not personally implement.

# 10. Suggested benchmark design

A serious research evaluation should avoid simply testing on arbitrary RNA pairs.

A stronger design could be:

1. obtain RNA families with trusted reference alignments;
2. divide families into development and held-out test sets;
3. optimize parameters only on development families;
4. evaluate separately at high, medium, and low sequence identity;
5. compare against sequence-only baselines;
6. report alignment accuracy as well as runtime;
7. inspect whether improvements arise consistently across RNA classes.

Potential experimental conditions could include:

| Method | Sequence score | RNA-FM embeddings | Learned parameters |
|---|---:|---:|---:|
| Conventional alignment | ✓ |  |  |
| RNA-FM only |  | ✓ |  |
| Hybrid score | ✓ | ✓ | \(lpha\) |
| Learned pair score | optional | ✓ | neural scoring function |

This converts the programming task into a genuine computational biology investigation.

# 11. Take-home message

For this RNA-FM alignment problem:

> **AI-assisted coding:**  
> *I know the alignment algorithm; help me implement it.*

> **Vibe coding:**  
> *I know what I want the RNA alignment program to do; build something that works.*

> **Agentic coding:**  
> *I know the research objective and constraints; systematically develop, test, benchmark, and improve an algorithm to address it.*

For scientific computing, the most promising direction is often **agentic coding with strong human scientific oversight**.

The AI can increasingly handle implementation, experimentation, debugging, and iteration, while the researcher remains responsible for the scientific question, validity of the benchmark, interpretation of results, and final conclusions.

# References and resources

- Chen, J. et al. **Interpretable RNA Foundation Model from Unannotated Data for Highly Accurate RNA Structure and Function Predictions.** RNA-FM.
- RNA-FM project repository: `https://github.com/ml4bio/RNA-FM`
- RNA-FM usage documentation: `https://ml4bio.github.io/RNA-FM/Usages/`

> Note: This notebook is designed as a conceptual and teaching example. The code snippets illustrate the architecture of an RNA-FM-based alignment system but do not constitute a complete validated alignment method.